# Cell 1 - Imports

In [1]:
from pathlib import Path
import datetime as dt
import numpy as np
import pandas as pd

# CELL 2 — Helper utilities

In [2]:
def must_exist(path, name):
    if not Path(path).exists():
        raise FileNotFoundError(f"{name} not found: {path}")

def show_df(df, n=5):
    display(df.head(n))

## 1) CONFIG (edit only this cell)

In [10]:
# =========================
# EDIT ONLY THIS BLOCK
# =========================

DRUG = "FLU"        # "FLU" or "PULV"

# -------- Base directories --------
BASE_DIR = Path("/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis")
DATA_DIR = BASE_DIR / "data"

# Genotype lives in ORANGE
GENO_DIR = Path("/orange/juannanzhou/MarginalEpistasis/data")

if DRUG not in {"FLU", "PULV"}:
    raise ValueError("DRUG must be 'FLU' or 'PULV'")

# -------- Choose test mode --------
TEST_MODE = input(
    "Choose epistasis test: [1] LD-pruned×QTL  [2] All×QTL : "
).strip() or "1"

TEST_MODE = int(TEST_MODE)
RUN_LABEL = {1: "ldpruned_x_qtl", 2: "all_x_qtl"}[TEST_MODE]

TODAY = dt.date.today().strftime("%Y%m%d")

# Single naming convention for ALL outputs (matches existing PLINK pipeline)
drug_code = {"FLU": "flu", "PULV": "pulv"}[DRUG]

# -------- Input files --------

# Genotype + index (ORANGE)
GENO_NPY = GENO_DIR / "geno_top_60.npy"
INDEX_TSV = GENO_DIR / "index_top_60.tsv"

# SNP metadata (BLUE)
SNP_INFO_TSV = DATA_DIR / "snp_info_complete.tsv"

# Fitness files (BLUE)
PHENO_TSV = {
    "FLU": DATA_DIR / "FLU_FMIC_fitness.tsv",
    "PULV": DATA_DIR / "PUL_FMIC_fitness.tsv",
}[DRUG]

# QTL interval files (BLUE)
QTL_RESULTS_TSV = {
    "FLU": DATA_DIR / "Fluconazole_QTL_genes_20260312.csv",
    "PULV": DATA_DIR / "Pulvinatal_QTL_genes_20260330.csv",
}[DRUG]

# Shared LD-pruned SNP list
LDPRUNED_SNPS_TXT = DATA_DIR / "LDPRUNED_snps.txt"

# -------- Output files --------
EXTRACT_SNPS_TXT = DATA_DIR / f"{drug_code}_{RUN_LABEL}_candidate_snps.extract.txt"
SET_FILE = DATA_DIR / f"{drug_code}_{RUN_LABEL}.set"

# -------- Print summary --------
print("DRUG:", DRUG)
print("RUN_LABEL:", RUN_LABEL)
print("GENO_NPY:", GENO_NPY)
print("INDEX_TSV:", INDEX_TSV)
print("PHENO_TSV:", PHENO_TSV)
print("QTL_RESULTS_TSV:", QTL_RESULTS_TSV)
print("LDPRUNED_SNPS_TXT:", LDPRUNED_SNPS_TXT)
print("SET_FILE:", SET_FILE)

Choose epistasis test: [1] LD-pruned×QTL  [2] All×QTL :  2


DRUG: FLU
RUN_LABEL: all_x_qtl
GENO_NPY: /orange/juannanzhou/MarginalEpistasis/data/geno_top_60.npy
INDEX_TSV: /orange/juannanzhou/MarginalEpistasis/data/index_top_60.tsv
PHENO_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/FLU_FMIC_fitness.tsv
QTL_RESULTS_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/Fluconazole_QTL_genes_20260312.csv
LDPRUNED_SNPS_TXT: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/LDPRUNED_snps.txt
SET_FILE: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/flu_all_x_qtl.set


# CELL 4 — Check required files

In [11]:
must_exist(GENO_NPY, "GENO_NPY")
must_exist(INDEX_TSV, "INDEX_TSV")
must_exist(SNP_INFO_TSV, "SNP_INFO_TSV")
must_exist(PHENO_TSV, "PHENO_TSV")
must_exist(QTL_RESULTS_TSV, "QTL_RESULTS_TSV")
must_exist(LDPRUNED_SNPS_TXT, "LDPRUNED_SNPS_TXT")

print("All required input files found!")

All required input files found!


# CELL 5 — Load genotype matrix

In [12]:
G = np.load(GENO_NPY)

# Robust index reader: take only first column, tolerate extra tabs/fields
index_ids = pd.read_csv(
    INDEX_TSV,
    sep=r"\s+|\t+",
    engine="python",
    header=None,
    usecols=[0],
    dtype=str
)[0].str.strip()

index_ids = index_ids[index_ids.notna() & (index_ids != "")]

print("Genotype matrix shape:", G.shape)
print("Index IDs:", len(index_ids))

if len(index_ids) != G.shape[0]:
    raise ValueError(f"Index/genotype mismatch: index={len(index_ids)} vs G={G.shape[0]}")

# (Optional) if you want a DataFrame for consistency downstream
index_df = pd.DataFrame({"ID": index_ids.values})
print("Index file shape (cleaned):", index_df.shape)

Genotype matrix shape: (59970, 41594)
Index IDs: 59970
Index file shape (cleaned): (59970, 1)


# CELL 6 — Load SNP metadata

In [13]:
snp = pd.read_csv(SNP_INFO_TSV, sep="\t")

print("SNP info shape:", snp.shape)

if G.shape[1] != snp.shape[0]:
    raise ValueError("Mismatch: genotype columns != SNP metadata rows")

SNP info shape: (41594, 12)


# CELL 7 — Load QTL interval table

In [14]:
qtl_df = pd.read_csv(QTL_RESULTS_TSV, sep=None, engine="python")

print("QTL table shape:", qtl_df.shape)
print("Columns:", list(qtl_df.columns))

QTL table shape: (45, 9)
Columns: ['interval_id', 'chrom', 'start', 'end', 'genes', 'genes_std', 'pleio_score', 'pleio_percentile', 'lod_max']


# CELL 8 — Build QTL SNP list + write PLINK files

In [19]:
required = {"chrom", "start", "end"}
missing = required - set(qtl_df.columns)
if missing:
    raise ValueError(f"Missing required QTL columns: {missing}")

for c in ["SNP", "Chromosome", "Position (bp)"]:
    if c not in snp.columns:
        raise ValueError(f"snp_info missing required column '{c}'")

# Clean types
qtl_df["chrom"] = qtl_df["chrom"].astype(int)
qtl_df["start"] = qtl_df["start"].astype(int)
qtl_df["end"] = qtl_df["end"].astype(int)

# Extract SNPs inside QTL intervals
qtl_snps = set()

for _, r in qtl_df.iterrows():
    hits = snp.loc[
        (snp["Chromosome"] == r["chrom"]) &
        (snp["Position (bp)"] >= r["start"]) &
        (snp["Position (bp)"] <= r["end"]),
        "SNP"
    ].astype(str).str.strip()

    qtl_snps.update(hits.tolist())

qtl_snps = {s for s in qtl_snps if s and s.lower() != "nan"}

print("QTL SNPs extracted:", len(qtl_snps))

all_snps = set(snp["SNP"].astype(str).str.strip())
all_snps = {s for s in all_snps if s and s.lower() != "nan"}

if TEST_MODE == 1:
    ld = pd.read_csv(LDPRUNED_SNPS_TXT, header=None, sep=r"\s+")[0].astype(str).str.strip()
    ldpruned_snps = set(ld.tolist()).intersection(all_snps)

    extract_snps = sorted(ldpruned_snps.union(qtl_snps))

    with open(SET_FILE, "w") as f:
        f.write("LDPRUNED\n")
        for s in sorted(ldpruned_snps):
            f.write(s + "\n")
        f.write("END\n")
        f.write("QTL\n")
        for s in sorted(qtl_snps):
            f.write(s + "\n")
        f.write("END\n")

elif TEST_MODE == 2:
    extract_snps = sorted(all_snps)

    with open(SET_FILE, "w") as f:
        f.write("QTL\n")
        for s in sorted(qtl_snps):
            f.write(s + "\n")
        f.write("END\n")

else:
    raise ValueError(f"Unsupported TEST_MODE: {TEST_MODE}")

pd.Series(extract_snps).to_csv(EXTRACT_SNPS_TXT, index=False, header=False)

print("Extract list written:", EXTRACT_SNPS_TXT)
print("Set file written:", SET_FILE)
print("Extract SNP count:", len(extract_snps))

QTL SNPs extracted: 2825
Extract list written: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/flu_all_x_qtl_candidate_snps.extract.txt
Set file written: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/flu_all_x_qtl.set
Extract SNP count: 41594


In [20]:
len(qtl_snps)

2825

## Quick Diagnostics

In [21]:
import pandas as pd

n_extract = pd.read_csv(EXTRACT_SNPS_TXT, header=None).shape[0]
print("Extract SNP count:", n_extract)
print("Expected (all SNPs):", len(all_snps))

# Also check set file size (QTL SNP count)
with open(SET_FILE, "r") as f:
    lines = [x.strip() for x in f if x.strip()]

print("Set file first line (set name):", lines[0])
print("Set file SNP count (excluding QTL + END):", len(lines) - 2)

Extract SNP count: 41594
Expected (all SNPs): 41594
Set file first line (set name): QTL
Set file SNP count (excluding QTL + END): 2825


# CELL 9 — Final sanity check

In [22]:
must_exist(EXTRACT_SNPS_TXT, "EXTRACT_SNPS_TXT")
must_exist(SET_FILE, "SET_FILE")

print("First 10 SNPs in extract file:")
print(pd.read_csv(EXTRACT_SNPS_TXT, header=None).head(10))

print("Pipeline input generation complete.")

First 10 SNPs in extract file:
          0
0      snp1
1     snp10
2    snp100
3   snp1000
4  snp10000
5  snp10001
6  snp10002
7  snp10003
8  snp10004
9  snp10005
Pipeline input generation complete.
